# 10 — Loss Functions

In the previous notebook, we learned how `nn.Module` organizes neural-network models.

Now we will study another essential part of training:

> **Loss functions**

A loss function measures how different a model's prediction is from the correct target.

During training:

$$
\text{Input}
\rightarrow
\text{Model}
\rightarrow
\text{Prediction}
\rightarrow
\text{Loss}
\rightarrow
\text{Backward Pass}
\rightarrow
\text{Parameter Update}
$$

The loss gives the model a numerical signal that says:

> **How wrong is the current prediction?**

## In this notebook, we will learn:

1. What is a loss function?
2. Regression vs classification losses
3. Mean Squared Error
4. Mean Absolute Error
5. Binary Cross Entropy
6. `BCEWithLogitsLoss`
7. Sigmoid and logits
8. Binary classification target shapes
9. Multi-label classification
10. Multi-class classification
11. Softmax intuition
12. `CrossEntropyLoss`
13. Target shapes and dtypes
14. Loss reduction
15. Choosing the correct loss
16. Common loss-function mistakes
17. Debugging loss problems
18. Practice exercises

## Main Goal

By the end of this notebook, you should be able to answer:

> **What should my model output, what should my target look like, and which loss function should I use?**

This is one of the most important practical skills in PyTorch.


In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

print("PyTorch version:", torch.__version__)


# 1. What Is a Loss Function?

A loss function compares:

- Model predictions
- Ground-truth targets

and produces a number measuring prediction error.

Conceptually:

$$
\boxed{
Loss = f(\text{prediction},\text{target})
}
$$

A smaller loss usually means the prediction is closer to the desired target.

During training, gradient descent tries to reduce this loss.


# 2. Why Do We Need a Loss Function?

Suppose a regression model predicts:

$$
\hat{y}=8
$$

but the correct target is:

$$
y=10
$$

We need a numerical way to describe the error.

For example:

$$
\hat{y}-y=8-10=-2
$$

A loss function converts prediction errors into a value that can be optimized.

Without a loss function, the model has no training objective.


# 3. Loss vs Metric

A **loss function** is normally used for optimization.

A **metric** is normally used to evaluate model performance in a human-interpretable way.

Examples:

$$
\begin{array}{|c|c|}
\hline
\textbf{Loss} & \textbf{Metric} \\
\hline
MSE & RMSE \\
\hline
BCE & Accuracy \\
\hline
CrossEntropy & Accuracy \\
\hline
CrossEntropy & F1\ Score \\
\hline
\end{array}
$$

The best training loss is not always the same quantity as the final evaluation metric.


# 4. Regression vs Classification

The correct loss depends heavily on the task.

## Regression

The target is usually a continuous numerical value.

Examples:

- Temperature
- House price
- Blood pressure
- Age
- Tumor size

Common regression losses:

- Mean Squared Error
- Mean Absolute Error

## Classification

The target represents a category or set of categories.

Examples:

- Healthy vs diseased
- Cat vs dog
- One of 10 digit classes
- Multiple labels for one image

Common classification losses:

- Binary Cross Entropy
- `BCEWithLogitsLoss`
- `CrossEntropyLoss`


# 5. Mean Squared Error — MSE

Mean Squared Error is one of the most common regression losses.

For $N$ examples:

$$
\boxed{
MSE=
\frac{1}{N}
\sum_{i=1}^{N}
(\hat{y}_i-y_i)^2
}
$$

where:

- $\hat{y}_i$ = prediction
- $y_i$ = target

The steps are:

1. Calculate prediction error
2. Square it
3. Average all squared errors


## MSE Example

Targets:

$$
y=
\begin{array}{|c|c|c|}
\hline
2 & 4 & 6 \\
\hline
\end{array}
$$

Predictions:

$$
\hat{y}=
\begin{array}{|c|c|c|}
\hline
1 & 5 & 4 \\
\hline
\end{array}
$$

Errors:

$$
\begin{array}{|c|c|c|}
\hline
-1 & 1 & -2 \\
\hline
\end{array}
$$

Squared errors:

$$
\begin{array}{|c|c|c|}
\hline
1 & 1 & 4 \\
\hline
\end{array}
$$

Therefore:

$$
MSE=
\frac{1+1+4}{3}
=
\boxed{2}
$$


In [ ]:
targets = torch.tensor([2.0, 4.0, 6.0])
predictions = torch.tensor([1.0, 5.0, 4.0])

errors = predictions - targets
squared_errors = errors ** 2
mse = squared_errors.mean()

print("Errors:", errors)
print("Squared errors:", squared_errors)
print("MSE:", mse)


# 6. `nn.MSELoss`

PyTorch provides:

`nn.MSELoss()`


In [ ]:
mse_loss = nn.MSELoss()

targets = torch.tensor([2.0, 4.0, 6.0])
predictions = torch.tensor([1.0, 5.0, 4.0])

loss = mse_loss(predictions, targets)

print("MSELoss:", loss)


# 7. Why Squaring Matters in MSE

Squaring the error has two important effects:

1. Positive and negative errors cannot cancel
2. Large errors receive a stronger penalty

For example:

$$
error=2
\Rightarrow
error^2=4
$$

but:

$$
error=10
\Rightarrow
error^2=100
$$

So MSE is relatively sensitive to large errors and outliers.


# 8. Visualizing Squared Error

The squared-error penalty grows quickly as the error becomes larger.


In [ ]:
errors = torch.linspace(-5, 5, 200)
squared_loss = errors ** 2

plt.figure(figsize=(8, 5))
plt.plot(errors.numpy(), squared_loss.numpy())
plt.xlabel("Prediction Error")
plt.ylabel("Squared Error")
plt.title("Squared Error Penalty")
plt.show()


# 9. Mean Absolute Error — MAE

Mean Absolute Error uses the absolute difference between prediction and target.

$$
\boxed{
MAE=
\frac{1}{N}
\sum_{i=1}^{N}
|\hat{y}_i-y_i|
}
$$

Unlike MSE, the error is not squared.


## MAE Example

Using the same errors:

$$
\begin{array}{|c|c|c|}
\hline
-1 & 1 & -2 \\
\hline
\end{array}
$$

Absolute errors:

$$
\begin{array}{|c|c|c|}
\hline
1 & 1 & 2 \\
\hline
\end{array}
$$

Therefore:

$$
MAE=
\frac{1+1+2}{3}
=
\frac{4}{3}
$$


In [ ]:
targets = torch.tensor([2.0, 4.0, 6.0])
predictions = torch.tensor([1.0, 5.0, 4.0])

absolute_errors = torch.abs(predictions - targets)
mae = absolute_errors.mean()

print("Absolute errors:", absolute_errors)
print("MAE:", mae)


# 10. `nn.L1Loss`

PyTorch calls Mean Absolute Error:

`nn.L1Loss()`


In [ ]:
mae_loss = nn.L1Loss()

loss = mae_loss(predictions, targets)

print("L1Loss / MAE:", loss)


# 11. MSE vs MAE

$$
\begin{array}{|c|c|c|}
\hline
\textbf{Property} & \textbf{MSE} & \textbf{MAE} \\
\hline
\text{Formula} & (\hat{y}-y)^2 & |\hat{y}-y| \\
\hline
\text{Large-error penalty} & \text{Strong} & \text{Linear} \\
\hline
\text{Outlier sensitivity} & \text{Higher} & \text{Lower} \\
\hline
\text{PyTorch} & nn.MSELoss & nn.L1Loss \\
\hline
\end{array}
$$

There is no universally best loss.

The correct choice depends on the problem and what kinds of errors matter.


In [ ]:
errors = torch.linspace(-5, 5, 200)

mse_values = errors ** 2
mae_values = torch.abs(errors)

plt.figure(figsize=(8, 5))
plt.plot(errors.numpy(), mse_values.numpy(), label="Squared error")
plt.plot(errors.numpy(), mae_values.numpy(), label="Absolute error")
plt.xlabel("Prediction Error")
plt.ylabel("Penalty")
plt.title("MSE vs MAE Penalty")
plt.legend()
plt.show()


# 12. Regression Output Shape

Suppose we predict one continuous value for each sample.

For a batch of 32 samples:

$$
prediction.shape=(32,\ 1)
$$

A clean target shape is usually:

$$
target.shape=(32,\ 1)
$$

Keeping prediction and target shapes aligned helps prevent accidental broadcasting.


In [ ]:
predictions = torch.randn(32, 1)
targets = torch.randn(32, 1)

criterion = nn.MSELoss()

loss = criterion(predictions, targets)

print("Prediction shape:", predictions.shape)
print("Target shape:", targets.shape)
print("Loss shape:", loss.shape)


# 13. Binary Classification

Binary classification means there are two possibilities.

Examples:

- Negative / Positive
- Healthy / Diseased
- No / Yes
- Class 0 / Class 1

A common convention is:

$$
y\in\{0,1\}
$$

For one sample:

$$
0=\text{negative class}
$$

$$
1=\text{positive class}
$$


# 14. What Is a Logit?

A binary classification model often produces one raw number called a:

> **Logit**

A logit can be any real number:

$$
-\infty < z < \infty
$$

Examples:

$$
\begin{array}{|c|}
\hline
-4.2 \\
\hline
0.0 \\
\hline
1.7 \\
\hline
5.3 \\
\hline
\end{array}
$$

A logit is **not yet a probability**.


# 15. Sigmoid Converts a Logit to a Probability

The sigmoid function is:

$$
\boxed{
\sigma(z)=\frac{1}{1+e^{-z}}
}
$$

Its output lies between:

$$
0
$$

and:

$$
1
$$

So we can convert a logit into a probability.


In [ ]:
logits = torch.tensor([-4.0, -1.0, 0.0, 1.0, 4.0])

probabilities = torch.sigmoid(logits)

print("Logits:", logits)
print("Probabilities:", probabilities)


# 16. Sigmoid Intuition

Some useful values:

$$
\begin{array}{|c|c|}
\hline
\textbf{Logit} & \textbf{Sigmoid Probability} \\
\hline
\text{Large negative} & \text{Near }0 \\
\hline
0 & 0.5 \\
\hline
\text{Large positive} & \text{Near }1 \\
\hline
\end{array}
$$

A decision threshold such as `0.5` can later convert probabilities into predicted classes.

Training loss and classification threshold are separate ideas.


In [ ]:
x = torch.linspace(-8, 8, 200)
y = torch.sigmoid(x)

plt.figure(figsize=(8, 5))
plt.plot(x.numpy(), y.numpy())
plt.axhline(0.5)
plt.axvline(0)
plt.xlabel("Logit")
plt.ylabel("Probability")
plt.title("Sigmoid")
plt.show()


# 17. Binary Cross Entropy

Binary Cross Entropy compares:

- A probability prediction
- A binary target

For one example:

$$
\boxed{
BCE=
-\left[
y\log(p)
+
(1-y)\log(1-p)
\right]
}
$$

where:

- $y\in\{0,1\}$
- $p$ is predicted probability


# 18. BCE When the Target Is 1

If:

$$
y=1
$$

then:

$$
BCE=-\log(p)
$$

If the model predicts:

$$
p\approx1
$$

loss is small.

If the model predicts:

$$
p\approx0
$$

loss becomes very large.

That is exactly what we want.


# 19. BCE When the Target Is 0

If:

$$
y=0
$$

then:

$$
BCE=-\log(1-p)
$$

If:

$$
p\approx0
$$

loss is small.

If:

$$
p\approx1
$$

loss is very large.


In [ ]:
bce = nn.BCELoss()

probabilities = torch.tensor([0.9, 0.2, 0.8])
targets = torch.tensor([1.0, 0.0, 1.0])

loss = bce(probabilities, targets)

print("BCE:", loss)


# 20. Important Requirement of `nn.BCELoss`

`nn.BCELoss()` expects **probabilities**, not raw logits.

Therefore a model would need:

$$
logits
\rightarrow
sigmoid
\rightarrow
BCELoss
$$

However, this is usually **not the preferred training pattern** in PyTorch.

A better option is:

`nn.BCEWithLogitsLoss()`


# 21. `BCEWithLogitsLoss`

`nn.BCEWithLogitsLoss()` combines:

- Sigmoid
- Binary Cross Entropy

into one numerically stable operation.

The recommended training pattern is:

$$
\boxed{
Raw\ Logits
\rightarrow
BCEWithLogitsLoss
}
$$

Do **not** apply sigmoid first when using `BCEWithLogitsLoss`.


In [ ]:
criterion = nn.BCEWithLogitsLoss()

logits = torch.tensor([2.0, -1.5, 0.8])
targets = torch.tensor([1.0, 0.0, 1.0])

loss = criterion(logits, targets)

print("BCEWithLogitsLoss:", loss)


# 22. `BCELoss` vs `BCEWithLogitsLoss`

$$
\begin{array}{|c|c|}
\hline
\textbf{BCELoss} & \textbf{BCEWithLogitsLoss} \\
\hline
\text{Input = probabilities} & \text{Input = raw logits} \\
\hline
\text{Requires sigmoid first} & \text{Includes sigmoid internally} \\
\hline
\text{Less preferred for training} & \text{Usually preferred} \\
\hline
\end{array}
$$

Recommended:

```python
logits = model(x)
loss = criterion(logits, targets)
```

where:

```python
criterion = nn.BCEWithLogitsLoss()
```


# 23. Why `BCEWithLogitsLoss` Is Preferred

Applying sigmoid and BCE separately can be less numerically stable for very large positive or negative logits.

`BCEWithLogitsLoss` combines the operations in a more stable mathematical form.

So during training:

> **Use raw logits with `BCEWithLogitsLoss`.**

During inference:

> **Apply sigmoid when you want probabilities.**


# 24. Binary Prediction Pipeline

Training:

$$
\boxed{
Input
\rightarrow
Model
\rightarrow
Logit
\rightarrow
BCEWithLogitsLoss
}
$$

Inference:

$$
\boxed{
Input
\rightarrow
Model
\rightarrow
Logit
\rightarrow
Sigmoid
\rightarrow
Probability
\rightarrow
Threshold
}
$$


In [ ]:
logits = torch.tensor([-2.0, 0.3, 1.8])

probabilities = torch.sigmoid(logits)
predictions = (probabilities >= 0.5).long()

print("Logits:", logits)
print("Probabilities:", probabilities)
print("Predicted classes:", predictions)


# 25. Binary Target Shape and Dtype

For binary classification with one output per sample:

Prediction logits:

$$
(batch,\ 1)
$$

Targets should commonly also be:

$$
(batch,\ 1)
$$

and use floating-point values:

$$
0.0,\ 1.0
$$

Example:


In [ ]:
logits = torch.randn(8, 1)

targets = torch.tensor([
    [1.0],
    [0.0],
    [1.0],
    [1.0],
    [0.0],
    [0.0],
    [1.0],
    [0.0]
])

criterion = nn.BCEWithLogitsLoss()

loss = criterion(logits, targets)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)
print("Target dtype:", targets.dtype)
print("Loss:", loss)


# 26. Common Binary Shape Mistake

Suppose:

$$
logits.shape=(32,\ 1)
$$

but:

$$
targets.shape=(32)
$$

These shapes are not the same.

For `BCEWithLogitsLoss`, it is best to explicitly align them.

For example:

```python
targets = targets.unsqueeze(1)
```

or make the model output shape `(32)` if that is your chosen convention.

The important rule is:

> **Prediction and target shapes should match for BCE-style losses.**


In [ ]:
targets = torch.randint(
    0,
    2,
    (32,)
).float()

print("Before:", targets.shape)

targets = targets.unsqueeze(1)

print("After:", targets.shape)


# 27. Multi-Label Classification

Multi-label classification is different from multi-class classification.

In multi-label classification, one sample can belong to **multiple classes at the same time**.

Example:

An image might contain:

- Cat = 1
- Dog = 0
- Person = 1

Target:

$$
\begin{array}{|c|c|c|}
\hline
1 & 0 & 1 \\
\hline
\end{array}
$$

For multi-label classification:

- Model outputs one logit per label
- Targets are commonly multi-hot floating-point vectors
- `BCEWithLogitsLoss` is a common choice


In [ ]:
logits = torch.randn(4, 3)

targets = torch.tensor([
    [1.0, 0.0, 1.0],
    [0.0, 1.0, 0.0],
    [1.0, 1.0, 0.0],
    [0.0, 0.0, 1.0]
])

criterion = nn.BCEWithLogitsLoss()

loss = criterion(logits, targets)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)
print("Loss:", loss)


# 28. Multi-Class Classification

Multi-class classification means:

> Each sample belongs to exactly one class among several possible classes.

Examples:

- Digit 0–9
- Cat / Dog / Horse
- One disease category among several mutually exclusive diagnoses

For 3 classes, a model may output three logits:

$$
\begin{array}{|c|c|c|}
\hline
2.1 & -0.4 & 1.3 \\
\hline
\end{array}
$$

Each output corresponds to one class.


# 29. Softmax Intuition

Softmax converts a vector of logits into probabilities that:

1. Are between `0` and `1`
2. Sum to `1`

For logits:

$$
z_1,z_2,\ldots,z_C
$$

softmax is:

$$
\boxed{
P(y=i)
=
\frac{e^{z_i}}
{\sum_{j=1}^{C}e^{z_j}}
}
$$


In [ ]:
logits = torch.tensor([2.1, -0.4, 1.3])

probabilities = torch.softmax(logits, dim=0)

print("Logits:", logits)
print("Probabilities:", probabilities)
print("Sum:", probabilities.sum())


# 30. Softmax Does Not Change the Ranking

If one class has the largest logit, it also has the largest softmax probability.

So class prediction can often be obtained directly from:

`argmax(logits)`

without calculating softmax first.


In [ ]:
logits = torch.tensor([2.1, -0.4, 1.3])

print("Argmax of logits:", logits.argmax())

probabilities = torch.softmax(logits, dim=0)

print("Argmax of probabilities:", probabilities.argmax())


# 31. Batch Softmax

For a batch:

$$
logits.shape=(batch,\ classes)
$$

we normally apply softmax across the class dimension:

`dim=1`


In [ ]:
logits = torch.randn(4, 3)

probabilities = torch.softmax(
    logits,
    dim=1
)

print("Logits shape:", logits.shape)
print("Probabilities shape:", probabilities.shape)

print("\\nRow sums:")
print(probabilities.sum(dim=1))


# 32. `CrossEntropyLoss`

For multi-class classification, PyTorch commonly uses:

`nn.CrossEntropyLoss()`

This loss expects:

- Raw logits
- Integer class indices

Do **not** apply softmax before `CrossEntropyLoss`.

Recommended training pattern:

$$
\boxed{
Raw\ Logits
\rightarrow
CrossEntropyLoss
}
$$


In [ ]:
criterion = nn.CrossEntropyLoss()

logits = torch.tensor([
    [2.0, 1.0, 0.1],
    [0.5, 2.5, 0.3],
    [1.0, 0.2, 3.0]
])

targets = torch.tensor([0, 1, 2])

loss = criterion(logits, targets)

print("Loss:", loss)


# 33. Multi-Class Target Representation

For 3 classes, the target for one sample is usually a **class index**.

For example:

$$
\begin{array}{|c|c|}
\hline
\textbf{Class Name} & \textbf{Target Index} \\
\hline
Class\ A & 0 \\
\hline
Class\ B & 1 \\
\hline
Class\ C & 2 \\
\hline
\end{array}
$$

For a batch of 5 samples:

$$
target.shape=(5)
$$

Example:

$$
\begin{array}{|c|c|c|c|c|}
\hline
0 & 2 & 1 & 1 & 0 \\
\hline
\end{array}
$$

The dtype should usually be:

`torch.long`


In [ ]:
targets = torch.tensor(
    [0, 2, 1, 1, 0],
    dtype=torch.long
)

print("Shape:", targets.shape)
print("Dtype:", targets.dtype)


# 34. Cross-Entropy Shape Rule

Suppose:

$$
batch=32
$$

and:

$$
classes=10
$$

Model logits:

$$
\boxed{(32,\ 10)}
$$

Targets:

$$
\boxed{(32)}
$$

Target values must be valid class indices such as:

$$
0,1,\ldots,9
$$


In [ ]:
logits = torch.randn(32, 10)

targets = torch.randint(
    low=0,
    high=10,
    size=(32,),
    dtype=torch.long
)

criterion = nn.CrossEntropyLoss()

loss = criterion(logits, targets)

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)
print("Targets dtype:", targets.dtype)
print("Loss:", loss)


# 35. `CrossEntropyLoss` Includes Softmax-Like Processing Internally

Conceptually, multi-class cross entropy involves converting logits into normalized class probabilities and penalizing low probability assigned to the correct class.

PyTorch combines the necessary operations in a numerically stable way.

Therefore:

> **Do not apply softmax before `CrossEntropyLoss`.**

Training:

$$
\boxed{
logits
\rightarrow
CrossEntropyLoss
}
$$

Inference:

$$
\boxed{
logits
\rightarrow
softmax
\rightarrow
probabilities
}
$$

if probabilities are needed.


# 36. Wrong Pattern — Softmax Before Cross Entropy

Avoid this during training:

```python
probabilities = torch.softmax(logits, dim=1)
loss = criterion(probabilities, targets)
```

`CrossEntropyLoss` expects raw logits.

Correct:

```python
loss = criterion(logits, targets)
```


In [ ]:
logits = torch.randn(8, 4)
targets = torch.randint(0, 4, (8,))

criterion = nn.CrossEntropyLoss()

correct_loss = criterion(
    logits,
    targets
)

print("Correct loss:", correct_loss)


# 37. Binary vs Multi-Class vs Multi-Label

This distinction is essential.

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Task} & \textbf{Output} & \textbf{Target} & \textbf{Common Loss} \\
\hline
\text{Binary} & 1\ logit & 0/1\ float & BCEWithLogitsLoss \\
\hline
\text{Multi-class} & C\ logits & class\ index & CrossEntropyLoss \\
\hline
\text{Multi-label} & C\ logits & C\ binary\ floats & BCEWithLogitsLoss \\
\hline
\end{array}
$$

Do not confuse multi-class and multi-label problems.


# 38. Example — Binary Classification

Architecture:

$$
features
\rightarrow
hidden
\rightarrow
1
$$

For batch size 16:

$$
logits.shape=(16,\ 1)
$$

Targets:

$$
targets.shape=(16,\ 1)
$$

Loss:

`BCEWithLogitsLoss`


In [ ]:
binary_model = nn.Sequential(
    nn.Linear(5, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.randn(16, 5)
targets = torch.randint(
    0,
    2,
    (16, 1)
).float()

logits = binary_model(X)

criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, targets)

print("Logits:", logits.shape)
print("Targets:", targets.shape)
print("Loss:", loss.item())


# 39. Example — Multi-Class Classification

Architecture:

$$
features
\rightarrow
hidden
\rightarrow
C
$$

Suppose:

$$
C=4
$$

Batch:

$$
16
$$

Then:

$$
logits.shape=(16,\ 4)
$$

Targets:

$$
targets.shape=(16)
$$

Loss:

`CrossEntropyLoss`


In [ ]:
multiclass_model = nn.Sequential(
    nn.Linear(5, 8),
    nn.ReLU(),
    nn.Linear(8, 4)
)

X = torch.randn(16, 5)
targets = torch.randint(
    0,
    4,
    (16,),
    dtype=torch.long
)

logits = multiclass_model(X)

criterion = nn.CrossEntropyLoss()
loss = criterion(logits, targets)

print("Logits:", logits.shape)
print("Targets:", targets.shape)
print("Target dtype:", targets.dtype)
print("Loss:", loss.item())


# 40. Example — Multi-Label Classification

Suppose there are 4 independent labels.

For each sample:

$$
target=
\begin{array}{|c|c|c|c|}
\hline
1 & 0 & 1 & 0 \\
\hline
\end{array}
$$

Model output:

$$
4
$$

logits per sample.

For batch size 16:

$$
logits.shape=(16,\ 4)
$$

Targets:

$$
targets.shape=(16,\ 4)
$$

Loss:

`BCEWithLogitsLoss`


In [ ]:
multilabel_model = nn.Sequential(
    nn.Linear(5, 8),
    nn.ReLU(),
    nn.Linear(8, 4)
)

X = torch.randn(16, 5)
targets = torch.randint(
    0,
    2,
    (16, 4)
).float()

logits = multilabel_model(X)

criterion = nn.BCEWithLogitsLoss()
loss = criterion(logits, targets)

print("Logits:", logits.shape)
print("Targets:", targets.shape)
print("Loss:", loss.item())


# 41. Target Dtypes

Target dtype depends on the loss.

A useful rule:

$$
\begin{array}{|c|c|}
\hline
\textbf{Loss} & \textbf{Typical Target Dtype} \\
\hline
MSELoss & torch.float32 \\
\hline
L1Loss & torch.float32 \\
\hline
BCEWithLogitsLoss & torch.float32 \\
\hline
CrossEntropyLoss & torch.long \\
\hline
\end{array}
$$

This difference causes many beginner errors.


# 42. Why Cross Entropy Uses Integer Targets

`CrossEntropyLoss` for standard multi-class classification expects each target to identify the correct class.

Example:

$$
target=2
$$

means:

> The correct class is class index 2.

It does not normally require you to manually create:

$$
\begin{array}{|c|c|c|}
\hline
0 & 0 & 1 \\
\hline
\end{array}
$$

for the common class-index use case.


# 43. Reduction — `mean`, `sum`, and `none`

Many PyTorch losses support a `reduction` argument.

Common values:

- `"mean"`
- `"sum"`
- `"none"`

Default is often:

`"mean"`


In [ ]:
predictions = torch.tensor([1.0, 5.0, 4.0])
targets = torch.tensor([2.0, 4.0, 6.0])

mean_loss = nn.MSELoss(
    reduction="mean"
)

sum_loss = nn.MSELoss(
    reduction="sum"
)

none_loss = nn.MSELoss(
    reduction="none"
)

print(
    "mean:",
    mean_loss(predictions, targets)
)

print(
    "sum:",
    sum_loss(predictions, targets)
)

print(
    "none:",
    none_loss(predictions, targets)
)


# 44. Understanding Reduction

For squared errors:

$$
\begin{array}{|c|c|c|}
\hline
1 & 1 & 4 \\
\hline
\end{array}
$$

`reduction="none"` returns:

$$
\begin{array}{|c|c|c|}
\hline
1 & 1 & 4 \\
\hline
\end{array}
$$

`reduction="sum"` returns:

$$
1+1+4=6
$$

`reduction="mean"` returns:

$$
\frac{6}{3}=2
$$


# 45. Why `reduction="none"` Can Be Useful

Keeping individual losses can be useful when you want to:

- Inspect per-sample losses
- Apply custom weighting
- Analyze difficult samples
- Build specialized objectives


In [ ]:
criterion = nn.BCEWithLogitsLoss(
    reduction="none"
)

logits = torch.tensor([
    2.0,
    -1.0,
    0.2
])

targets = torch.tensor([
    1.0,
    0.0,
    1.0
])

individual_losses = criterion(
    logits,
    targets
)

print(individual_losses)


# 46. Class Imbalance and `pos_weight`

For imbalanced binary or multi-label problems, `BCEWithLogitsLoss` supports:

`pos_weight`

This can increase the contribution of positive examples.

For example:

```python
criterion = nn.BCEWithLogitsLoss(
    pos_weight=...
)
```

This is useful when positive examples are much rarer than negative examples.

Do not choose weights blindly.

They should be based on the training data and the objective of the problem.


In [ ]:
logits = torch.tensor([
    -1.0,
    0.5,
    1.2,
    -0.3
])

targets = torch.tensor([
    0.0,
    0.0,
    1.0,
    0.0
])

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(3.0)
)

loss = criterion(logits, targets)

print("Weighted BCE loss:", loss)


# 47. Classification Threshold Is Not the Training Loss

For binary classification:

`BCEWithLogitsLoss`

trains the model using logits and targets.

The classification threshold is typically applied later to probabilities.

Example:

$$
p\ge0.5
\Rightarrow
class=1
$$

The best operational threshold does not have to be `0.5`.

Threshold selection depends on the desired tradeoff between errors such as false positives and false negatives.

This is especially important in medical classification.


# 48. Choosing the Correct Loss

A useful decision table:

$$
\begin{array}{|c|c|c|c|}
\hline
\textbf{Problem} & \textbf{Model Output} & \textbf{Target} & \textbf{Loss} \\
\hline
\text{Regression} & continuous & float & MSELoss/L1Loss \\
\hline
\text{Binary} & 1\ logit & float\ 0/1 & BCEWithLogitsLoss \\
\hline
\text{Multi-class} & C\ logits & long\ class\ index & CrossEntropyLoss \\
\hline
\text{Multi-label} & C\ logits & C\ floats\ 0/1 & BCEWithLogitsLoss \\
\hline
\end{array}
$$

Memorize the logic, not just the names.


# 49. A Practical Loss-Selection Checklist

Before choosing a loss, ask:

1. Is the task regression or classification?
2. If classification, is it binary, multi-class, or multi-label?
3. What shape does the model output?
4. What shape does the target have?
5. What dtype does the loss expect?
6. Does the loss expect logits or probabilities?
7. Is class imbalance important?
8. What evaluation metric matters after training?


# 50. Common Mistake — Sigmoid Before `BCEWithLogitsLoss`

Incorrect:

```python
probabilities = torch.sigmoid(logits)
loss = criterion(probabilities, targets)
```

when:

```python
criterion = nn.BCEWithLogitsLoss()
```

Why?

Because `BCEWithLogitsLoss` already includes sigmoid internally.

Correct:

```python
loss = criterion(logits, targets)
```


# 51. Common Mistake — Softmax Before `CrossEntropyLoss`

Incorrect:

```python
probabilities = torch.softmax(
    logits,
    dim=1
)

loss = criterion(
    probabilities,
    targets
)
```

Correct:

```python
loss = criterion(
    logits,
    targets
)
```

`CrossEntropyLoss` expects raw logits.


# 52. Common Mistake — Wrong Target Dtype

For:

`CrossEntropyLoss`

a common target dtype is:

`torch.long`

For:

`BCEWithLogitsLoss`

targets should usually be floating point.


In [ ]:
binary_targets = torch.tensor(
    [0, 1, 1, 0]
).float()

multiclass_targets = torch.tensor(
    [0, 2, 1, 2]
).long()

print(
    "Binary dtype:",
    binary_targets.dtype
)

print(
    "Multi-class dtype:",
    multiclass_targets.dtype
)


# 53. Common Mistake — Wrong Output Size

For 5-class multi-class classification, the final layer commonly needs:

$$
5
$$

output logits.

Example:

`nn.Linear(hidden_features, 5)`

For binary classification with `BCEWithLogitsLoss`, one common design is:

`nn.Linear(hidden_features, 1)`

The output architecture must match the task and loss.


# 54. Common Mistake — Prediction/Target Shape Mismatch

Always print:

```python
print(logits.shape)
print(targets.shape)
```

before debugging anything more complicated.

Shape mistakes are extremely common.


In [ ]:
logits = torch.randn(8, 1)
targets = torch.randint(
    0,
    2,
    (8, 1)
).float()

print("Logits:", logits.shape)
print("Targets:", targets.shape)


# 55. Common Mistake — Using Accuracy as the Loss

Accuracy is not normally used as a training loss because operations such as hard thresholding and `argmax` are not suitable as smooth optimization objectives.

We usually train with a differentiable loss such as:

- BCE
- Cross entropy
- MSE

and evaluate with metrics such as:

- Accuracy
- Precision
- Recall
- F1
- AUROC
- Sensitivity
- Specificity


# 56. Common Mistake — Calling `.item()` Before `backward()`

This is incorrect:

```python
loss_value = loss.item()
loss_value.backward()
```

`.item()` converts the tensor to a Python number and disconnects it from Autograd.

Correct:

```python
loss.backward()

loss_value = loss.item()
```

Use `.item()` for logging after preserving the tensor loss for backpropagation.


# 57. One Binary Training Step

Let's connect model, logits, loss, and Autograd.


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 1)
)

X = torch.randn(16, 4)

targets = torch.randint(
    0,
    2,
    (16, 1)
).float()

criterion = nn.BCEWithLogitsLoss()

logits = model(X)

loss = criterion(
    logits,
    targets
)

loss.backward()

print("Logits shape:", logits.shape)
print("Loss:", loss.item())


# 58. One Multi-Class Training Step


In [ ]:
torch.manual_seed(42)

model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 3)
)

X = torch.randn(16, 4)

targets = torch.randint(
    0,
    3,
    (16,)
).long()

criterion = nn.CrossEntropyLoss()

logits = model(X)

loss = criterion(
    logits,
    targets
)

loss.backward()

print("Logits shape:", logits.shape)
print("Targets shape:", targets.shape)
print("Loss:", loss.item())


# 59. Converting Multi-Class Logits Into Predictions

For class predictions:

`argmax(dim=1)`

is commonly used.


In [ ]:
with torch.no_grad():
    predicted_classes = logits.argmax(
        dim=1
    )

print("Predicted classes:")
print(predicted_classes)


If class probabilities are needed:


In [ ]:
with torch.no_grad():
    probabilities = torch.softmax(
        logits,
        dim=1
    )

print("Probability shape:", probabilities.shape)
print("First sample probabilities:")
print(probabilities[0])
print("Sum:", probabilities[0].sum())


# 60. Converting Binary Logits Into Predictions

For binary classification:

1. Apply sigmoid
2. Apply threshold


In [ ]:
binary_logits = torch.tensor([
    [-2.0],
    [0.3],
    [1.8],
    [0.0]
])

probabilities = torch.sigmoid(
    binary_logits
)

predictions = (
    probabilities >= 0.5
).long()

print("Probabilities:")
print(probabilities)

print("\\nPredicted classes:")
print(predictions)


# 61. Loss-Function Debugging Checklist

When a loss gives an error or behaves strangely, inspect:

- Prediction shape
- Target shape
- Prediction dtype
- Target dtype
- Raw logits vs probabilities
- Number of output units
- Class-index range
- Reduction setting

Then ask:

1. What exact task am I solving?
2. Does my loss match the task?
3. Does my model output the expected number of values?
4. Does the loss expect logits or probabilities?
5. Are prediction and target shapes compatible?
6. Is the target dtype correct?
7. Are class indices valid?
8. Did I accidentally apply sigmoid or softmax twice?


# 62. Practice Exercises

Try solving these before looking at the solutions.

## Exercise 1

Calculate MSE for:

Targets:

$$
\begin{array}{|c|c|c|}
\hline
1 & 3 & 5 \\
\hline
\end{array}
$$

Predictions:

$$
\begin{array}{|c|c|c|}
\hline
2 & 2 & 7 \\
\hline
\end{array}
$$

## Exercise 2

Calculate MAE for the same values.

## Exercise 3

Create a binary classifier with 6 input features and one output logit.

Choose the correct loss.

## Exercise 4

Create targets for binary classification with shape:

$$
(32,\ 1)
$$

and the correct dtype.

## Exercise 5

Create a 5-class classifier with 10 input features.

What should the final layer output size be?

## Exercise 6

For a 5-class classifier with batch size 32, give the correct:

- Logit shape
- Target shape
- Target dtype

## Exercise 7

Create a multi-label classifier with 4 labels.

What loss should you use?

## Exercise 8

Explain why sigmoid should not be applied before `BCEWithLogitsLoss`.

## Exercise 9

Explain why softmax should not be applied before `CrossEntropyLoss`.

## Exercise 10

Use `reduction="none"` and inspect individual MSE values.


# 63. Shape and Dtype Challenges

Answer before running code.

## Challenge 1

Binary classification:

$$
batch=64
$$

One output logit per sample.

What are the recommended:

- Logit shape
- Target shape
- Target dtype
- Loss

## Challenge 2

Multi-class classification:

$$
batch=64
$$

$$
classes=7
$$

What are:

- Logit shape
- Target shape
- Target dtype
- Loss

## Challenge 3

Multi-label classification:

$$
batch=64
$$

$$
labels=7
$$

What are:

- Logit shape
- Target shape
- Target dtype
- Loss

## Challenge 4

Regression with 3 continuous outputs per sample and batch size 32.

What could the prediction and target shapes be?

Which losses from this notebook could be appropriate?

## Challenge 5

A model outputs:

$$
(16,\ 10)
$$

but targets are floating one-hot vectors of shape:

$$
(16,\ 10)
$$

You intended to use the standard class-index form of `CrossEntropyLoss`.

What should you reconsider about the target representation?


# 64. Exercise Solutions


In [ ]:
# Exercise 1
targets = torch.tensor([
    1.0,
    3.0,
    5.0
])

predictions = torch.tensor([
    2.0,
    2.0,
    7.0
])

mse = (
    (predictions - targets) ** 2
).mean()

print("Exercise 1 MSE:", mse)

# Exercise 2
mae = torch.abs(
    predictions - targets
).mean()

print("Exercise 2 MAE:", mae)

# Exercise 3
binary_model = nn.Linear(
    6,
    1
)

binary_loss = nn.BCEWithLogitsLoss()

print(
    "Exercise 3 output features:",
    binary_model.out_features
)

print(
    "Exercise 3 loss:",
    binary_loss
)

# Exercise 4
binary_targets = torch.randint(
    0,
    2,
    (32, 1)
).float()

print(
    "Exercise 4 shape:",
    binary_targets.shape
)

print(
    "Exercise 4 dtype:",
    binary_targets.dtype
)

# Exercise 5
multiclass_model = nn.Linear(
    10,
    5
)

print(
    "Exercise 5 output features:",
    multiclass_model.out_features
)

# Exercise 6
logits = torch.randn(
    32,
    5
)

targets = torch.randint(
    0,
    5,
    (32,),
    dtype=torch.long
)

print(
    "Exercise 6 logits:",
    logits.shape
)

print(
    "Exercise 6 targets:",
    targets.shape
)

print(
    "Exercise 6 dtype:",
    targets.dtype
)

# Exercise 7
multilabel_model = nn.Linear(
    10,
    4
)

multilabel_loss = nn.BCEWithLogitsLoss()

print(
    "Exercise 7:",
    multilabel_model,
    multilabel_loss
)

# Exercise 10
criterion = nn.MSELoss(
    reduction="none"
)

print(
    "Exercise 10:",
    criterion(
        predictions,
        targets=torch.tensor(
            [1.0, 3.0, 5.0]
        )
    )
)


# 65. Key Takeaways

In this notebook, we learned:

- What a loss function is
- Loss vs metric
- Regression vs classification
- Mean Squared Error
- Mean Absolute Error
- `nn.MSELoss`
- `nn.L1Loss`
- Binary classification
- Logits
- Sigmoid
- Binary Cross Entropy
- `BCELoss`
- `BCEWithLogitsLoss`
- Binary target shapes and dtypes
- Multi-label classification
- Multi-class classification
- Softmax
- `CrossEntropyLoss`
- Class-index targets
- Loss reduction
- `pos_weight`
- Choosing the correct loss
- Common shape and dtype mistakes

The most important patterns are:

## Regression

$$
\boxed{
continuous\ prediction
\rightarrow
MSELoss/L1Loss
}
$$

## Binary Classification

$$
\boxed{
1\ raw\ logit
\rightarrow
BCEWithLogitsLoss
}
$$

## Multi-Class Classification

$$
\boxed{
C\ raw\ logits
\rightarrow
CrossEntropyLoss
}
$$

## Multi-Label Classification

$$
\boxed{
C\ raw\ logits
\rightarrow
BCEWithLogitsLoss
}
$$


# 66. Check Your Understanding

Before moving forward, make sure you can answer these without searching:

1. What does a loss function measure?
2. What is the difference between a loss and a metric?
3. What is MSE?
4. What is MAE?
5. Why is MSE more sensitive to large errors?
6. What is binary classification?
7. What is a logit?
8. What does sigmoid do?
9. What does `BCELoss` expect as input?
10. What does `BCEWithLogitsLoss` expect?
11. Why is `BCEWithLogitsLoss` usually preferred?
12. What should binary target dtype usually be?
13. What is multi-label classification?
14. What is multi-class classification?
15. What does softmax do?
16. What does `CrossEntropyLoss` expect?
17. Why should we not softmax before `CrossEntropyLoss`?
18. What shape should multi-class targets normally have?
19. What dtype should class-index targets have?
20. What is the difference between multi-class and multi-label classification?
21. What does `reduction="none"` do?
22. How do you choose the correct loss function?
23. Why should prediction and target shapes be inspected first when debugging?


# Next Notebook

# 11 — Optimizers and Gradient Descent

In the next notebook, we will study:

- Gradient descent intuition
- Learning rate
- Parameter updates
- Stochastic Gradient Descent
- `torch.optim.SGD`
- Momentum
- Adam
- `torch.optim.Adam`
- `optimizer.zero_grad()`
- `loss.backward()`
- `optimizer.step()`
- Weight decay
- Comparing SGD and Adam
- Learning-rate effects
- Common optimizer mistakes
- Building a clean optimization loop
